In [5]:
import torch
import pandas as pd
from pathlib import Path

# ==========================================
# 1. Configurazione Parametri
# ==========================================
WINDOW_SIZE = 724
PT_FILE_PATH = Path('..\\generated_matrices\\data_00_20_w724_s10\\VAE_20dim_cholesky_06_loss\\data_00_20_w724_s1\\forecast_scenarios_100\\forecast_scenarios_100.pt')
CSV_FILE_PATH = Path('..\\data\\processed\\data_00_20\\log_returns_data_00_20.csv')  # <-- Inserisci il nome corretto del tuo file CSV
DATE_COLUMN = 'timestamp'                  # <-- Inserisci il nome corretto della colonna delle date

# ==========================================
# 2. Caricamento Dati
# ==========================================
print("Caricamento del file CSV...")
# Leggiamo il CSV. Se le date sono nella prima colonna e non hanno un'intestazione, usa header=None
df = pd.read_csv(CSV_FILE_PATH)

# Assicuriamoci che la colonna date sia nel formato corretto
df[DATE_COLUMN] = pd.to_datetime(df[DATE_COLUMN])
# Estraiamo la lista delle date (puoi formattarle come stringhe per comodità)
dates_list = df[DATE_COLUMN].dt.strftime('%Y-%m-%d').tolist()

print("Caricamento del file .pt...")
# Carichiamo il tensore/dizionario PyTorch
pt_data = torch.load(PT_FILE_PATH, weights_only=False)
# Estraiamo la lista degli indici
indices = pt_data['indices']

# ==========================================
# 3. Creazione del Mapping
# ==========================================
print(f"Creazione del mapping per {len(indices)} indici (Window Size: {WINDOW_SIZE})...")

# Dizionario che conterrà il risultato: { indice: ('data_inizio', 'data_fine') }
mapping_date = {}

for idx in indices:
    # L'indice rappresenta il punto di partenza della finestra.
    # L'ultimo elemento della finestra sarà a idx + WINDOW_SIZE - 1
    end_idx = idx + WINDOW_SIZE - 1
    
    # Controllo di sicurezza per evitare IndexError nel caso in cui la finestra superi i dati disponibili
    if end_idx < len(dates_list):
        start_date = dates_list[idx]
        end_date = dates_list[end_idx]
        mapping_date[idx] = (start_date, end_date)
    else:
        print(f"⚠️ Attenzione: l'indice {idx} produce una finestra che supera la lunghezza del CSV (Max righe: {len(dates_list)}).")

# ==========================================
# 4. Verifica dei Risultati
# ==========================================
print("\n--- Primi 5 risultati del mapping ---")
for i, (k, v) in enumerate(mapping_date.items()):
    if i >= 5: break
    print(f"Matrice all'indice {k:4d} --> Data Inizio: {v[0]} | Data Fine: {v[1]}")

print("Ultimi 5 risultati del mapping ---")
for i, (k, v) in enumerate(list(mapping_date.items())[-5:]):
    print(f"Matrice all'indice {k:4d} --> Data Inizio: {v[0]} | Data Fine: {v[1]}")
# (Opzionale) Trasformare il mapping in un DataFrame per visualizzarlo o salvarlo meglio
df_mapping = pd.DataFrame.from_dict(mapping_date, orient='index', columns=['Data_Inizio', 'Data_Fine'])
df_mapping.index.name = 'Indice_Matrice'

# Salva il risultato se necessario
df_mapping.to_csv(Path('..\\data\\processed\\data_00_20\\mapping_matrici_date.csv'))

Caricamento del file CSV...
Caricamento del file .pt...
Creazione del mapping per 4124 indici (Window Size: 724)...

--- Primi 5 risultati del mapping ---
Matrice all'indice  486 --> Data Inizio: 2001-10-11 | Data Fine: 2004-08-26
Matrice all'indice  487 --> Data Inizio: 2001-10-12 | Data Fine: 2004-08-27
Matrice all'indice  488 --> Data Inizio: 2001-10-15 | Data Fine: 2004-08-30
Matrice all'indice  489 --> Data Inizio: 2001-10-16 | Data Fine: 2004-08-31
Matrice all'indice  490 --> Data Inizio: 2001-10-17 | Data Fine: 2004-09-01
Ultimi 5 risultati del mapping ---
Matrice all'indice 4605 --> Data Inizio: 2018-02-22 | Data Fine: 2021-01-06
Matrice all'indice 4606 --> Data Inizio: 2018-02-23 | Data Fine: 2021-01-07
Matrice all'indice 4607 --> Data Inizio: 2018-02-26 | Data Fine: 2021-01-08
Matrice all'indice 4608 --> Data Inizio: 2018-02-27 | Data Fine: 2021-01-11
Matrice all'indice 4609 --> Data Inizio: 2018-02-28 | Data Fine: 2021-01-12
